## Building AI agent

The agent is comprised of a router using OpenAI function calling, and a set of three tools: a database lookup tool, a data analysis tool, and a code generator to create graphs.

The agent can lookup information from a local file, perform analysis on that information, and graph results. The example local file is a log of transactions at a local store. The agent can help the store owners understand trends and anomalies in their sales data.

In [1]:
from openai import OpenAI
import pandas as pd
import json
import duckdb
from pydantic import BaseModel, Field
from IPython.display import Markdown
from helper import get_openai_api_key

### duckdb:
it allows the agent to execute fast, complex SQL queries directly on local files (Parquet, CSV, JSON) or dataframes without needing a dedicated database server running in the background.

### pydantic:
When you build external tools for an agent to call, you use Pydantic to define the exact JSON schema the LLM must return. This ensures that the arguments passed to your downstream Python functions are strictly validated and correctly typed before execution, preventing the agent from crashing the workflow with hallucinated or malformed parameters.
1. BaseModel: By creating a class that inherits from BaseModel, you map out the expected attributes using standard Python type hints. It acts as the engine that automatically parses incoming data (like a JSON string from an LLM) and validates it against your defined types.
2. Field: Field is a function used to add constraints, metadata, and validation rules to specific attributes inside your BaseModel that go beyond basic type hints.

## setting up the openai client

In [2]:
openai_api = get_openai_api_key()
client = OpenAI(api_key = openai_api)

model = "gpt-4o-mini"

In [3]:
transaction_data = "data/Store_Sales_Price_Elasticity_Promotions_Data.parquet"

## parquet files:

it is an open-source, column-oriented binary file format designed for highly efficient storage and retrieval of large-scale datasets. Unlike traditional formats like CSV or JSON that store data row by row, Parquet groups and stores data column by column.


Row-Based vs. Columnar Storage

To understand why Parquet is powerful, consider a simple table containing customer purchases:

| Customer ID | Purchase Date | Amount | Country |
|---|---|---|---|
| 101 | 2026-10-01 | \$50.00 | USA |
| 102 | 2026-10-02 | \$20.00 | UK |

1. Row-Based (CSV/JSON): Stores data as 101, 2026-10-01, 50.00, USA, 102, 2026-10-02, 20.00, UK. If a query only requests the total of all Amounts, the system must still read every single customer record from disk.
2. Columnar (Parquet): Stores data as 101, 102, then 2026-10-01, 2026-10-02, then 50.00, 20.00, and so on. A query requesting total Amounts isolates and reads only that column, entirely skipping the others.

## defining the tools:

#### tool1:
1. First, it creates the SQL table from a local file, if not already done.
2. Second, it translates the original prompt into an sql query (using an LLM call).
3. Finally, it runs that query against the database.
    

In [4]:
sql_prompt = """
Generate an sql query based on the prompt. Do not reply with anything except the sql query.
The prompt is: {prompt},
the available colums are: {columns},
the table name is: {table_name}
"""

In [5]:
def generate_sql_query(prompt: str, columns: list, table_name: str) -> str:
    """generate an sql query based on the prompt"""
    
    formatted_prompt = sql_prompt.format(prompt = prompt, columns = columns, table_name = table_name)
    
    response = client.chat.completions.create(
        model = model,
        messages = [{"role": "user", "content": formatted_prompt}],
    )
    
    return response.choices[0].message.content

In [6]:
def lookup_sales_data(prompt: str) -> str:
    """implementation of sales data lookup from parquet file using sql"""
    
    try:
        table_name = "sales"
        
        #load the parquet file content into duckdb table
        df = pd.read_parquet(transaction_data)
        duckdb.sql(f"CREATE TABLE IF NOT EXISTS {table_name} AS SELECT * FROM df") #this creates a new table if not already exists then loads all the data from df into the table it just created
        
        #get the executable sql query
        sql_query = generate_sql_query(prompt, df.columns, table_name)
        sql_query = sql_query.strip()
        sql_query = sql_query.replace("```sql", "").replace("```", "")
        
        #execute the sql query on the newly created table
        result = duckdb.query(sql_query).df()
        
        return result.to_string()
    
    except Exception as e:
        return f"error accessing the data: {str(e)}"

In [7]:
e1 = lookup_sales_data("show me all the sales for the store 1320 on november 1st, 2021")
print(e1)

    Store_Number  SKU_Coded  Product_Class_Code  Sold_Date  Qty_Sold  Total_Sale_Value  On_Promo
0           1320    6173050               22875 2021-11-01         1          4.990000         0
1           1320    6174250               22875 2021-11-01         1          0.890000         0
2           1320    6176200               22975 2021-11-01         2         99.980003         0
3           1320    6176800               22800 2021-11-01         1         14.970000         0
4           1320    6177250               22975 2021-11-01         1          6.890000         0
5           1320    6177300               22800 2021-11-01         1          9.990000         0
6           1320    6177350               22800 2021-11-01         2         16.980000         0
7           1320    6177700               22875 2021-11-01         1          3.190000         0
8           1320    6178000               22875 2021-11-01         2          6.380000         0
9           1320    6178250   

### tool2:

The second tool can analyze the returned data and display conclusions to users.

In [8]:
data_analysis_prompt = """
analyze the following data: {data},
your job is to answer the following question: {prompt}
"""

In [9]:
def analyze_sales_data(prompt: str, data: str) -> str:
    """llm-powered sales data analysis"""
    
    formatted_prompt = data_analysis_prompt.format(data = data, prompt = prompt)
    
    response = client.chat.completions.create(
        model = model,
        messages = [{"role": "user", "content": formatted_prompt}]
    )
    
    analysis = response.choices[0].message.content
    return analysis if analysis else "no analysis could be made"

In [10]:
print(analyze_sales_data("what trends do you see in this data?", e1))

The provided dataset contains sales information for various products at a specific store on a single day (November 1, 2021). The relevant fields include `Store_Number`, `SKU_Coded`, `Product_Class_Code`, `Sold_Date`, `Qty_Sold`, `Total_Sale_Value`, and `On_Promo`. While the dataset is limited to one day, several trends can be observed.

### Trends Observed in the Data:

1. **Product Diversity**:
   - The data includes a variety of products identified by different `SKU_Coded` values and grouped into different `Product_Class_Code` categories (e.g., 22800, 22975, 24400, etc.). This indicates that the store carries a diverse range of products.

2. **Sales Quantity**:
   - There are instances of different quantities sold (e.g., 1, 2, 3, 5), with certain products experiencing higher volume sales than others. For example, `SKU_Coded` 6184100 sold 3 units, which suggests stronger demand for certain products on that day.

3. **Total Sale Value Variation**:
   - The `Total_Sale_Value` varies sig

### tool3:

The third tool generates python code to create the requested graphs from the returned data of the first tool. It consists of two steps:
1. First, it creates the chart configuration: chart type, title, data, lables for x-axis and y-axis (using an LLM call).
2. Second, it generates the python code based on the chart configuration of the first step (using an LLM call).

In [11]:
chart_config_prompt = """
Generate a chart configuration based on the data: {data},
the goal is to show: {visualization_goal}
"""

In [12]:
class VisualizationConfig(BaseModel):
    chart_type: str = Field(..., description = "type of chart to generate")
    x_axis: str = Field(..., description = "name of the x-axis column")
    y_axis: str = Field(..., description = "name of the y-axis column")
    title: str = Field(..., description = "title of the chart")

In [13]:
def extract_chart_config(data: str, visualization_goal: str) -> dict:
    """generate chart visualization configuration
    
    args: 
        data: string containing data to visualize
        visualization_goal: description of what the visualization should show
        
    returns:
        dictionary containing chart configuration
    """
    
    formatted_prompt = chart_config_prompt.format(data = data, visualization_goal = visualization_goal)
    
    response = client.beta.chat.completions.parse(
        model = model,
        messages = [{"role": "user", "content": formatted_prompt}],
        response_format = VisualizationConfig,
    )
    
    try:
        content = response.choices[0].message.parsed
        
        return {
            "chart_type": content.chart_type,
            "x_axis": content.x_axis,
            "y_axis": content.y_axis,
            "title": content.title,
            "data": data
        }
    
    except Exception:
        return {
            "chart_type": "line",
            "x_axis": "data",
            "y_axis": "value",
            "title": visualization_goal,
            "data": data
        }

In [14]:
create_chart_prompt = """
Write a python code to create a chart based on the following configuration, return only code, no other text.
Configuration: {config} 
"""

In [15]:
def create_chart(config: dict) -> str:
    """create a chart based on the configuration"""
    
    formatted_prompt = create_chart_prompt.format(config = config)
    response = client.chat.completions.create(
        model = model,
        messages = [{"role": "user", "content": formatted_prompt}],
    )
    
    code = response.choices[0].message.content
    code = code.strip()
    code = code.replace("```python", "").replace("```", "")
    
    return code

In [16]:
def generate_visualization(data: str, visualization_goal: str) -> str:
    "generate visualization based on the data and goal"
    
    config = extract_chart_config(data, visualization_goal)
    code = create_chart(config)
    return code

In [17]:
code = generate_visualization(e1, "A bar chart of sales by product SKU. Put the product SKU on the x-axis and the sales on the y-axis.")
print(code)


import pandas as pd
import matplotlib.pyplot as plt

data = {
    'Store_Number': [1320]*30,
    'SKU_Coded': [6173050, 6174250, 6176200, 6176800, 6177250, 6177300, 6177350, 6177700,
                  6178000, 6178250, 6179250, 6179300, 6179400, 6179450, 6179500, 6179750,
                  6180550, 6182050, 6183750, 6184100, 6188550, 6190050, 6190150, 6190200,
                  6190250, 6190350, 6190400, 6193750, 6195350, 6195800],
    'Product_Class_Code': [22875, 22875, 22975, 22800, 22975, 22800, 22800, 22875,
                           22875, 22800, 24400, 22800, 24400, 24400, 24400, 22800,
                           22975, 22975, 22850, 22975, 22950, 24425, 24425, 24425,
                           24425, 22950, 22950, 22875, 24375, 22850],
    'Sold_Date': ['2021-11-01']*30,
    'Qty_Sold': [1, 1, 2, 1, 1, 1, 2, 1, 2, 1, 1, 2, 2, 1, 1, 2,
                 1, 1, 3, 3, 2, 5, 1, 1, 1, 1, 1, 1, 1, 1, 3],
    'Total_Sale_Value': [4.99, 0.89, 99.98, 14.97, 6.89, 9.99, 16.98, 3.19,
    

LLMs are non-deterministic, they are prone to fail if you let them write out 30 or more numbers accurately without missing a single comma.


A robust agent avoids this by keeping the data on the hard drive. Instead of passing the raw data string to the LLM and saying "make a chart using these numbers," you adjust the system prompt or tool configuration to say: "Write a script that loads the data directly from data/Store_Sales_Price_Elasticity_Promotions_Data.parquet to create the chart."

But for now we are going to keep it simple.

## Defining the router

In [18]:
# Define tools/functions that can be called by the model
tools = [
    {
        "type": "function",
        "function": {
            "name": "lookup_sales_data",
            "description": "Look up data from Store Sales Price Elasticity Promotions dataset",
            "parameters": {
                "type": "object",
                "properties": {
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "analyze_sales_data", 
            "description": "Analyze sales data to extract insights",
            "parameters": {
                "type": "object",
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "prompt": {"type": "string", "description": "The unchanged prompt that the user provided."}
                },
                "required": ["data", "prompt"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_visualization",
            "description": "Generate Python code to create data visualizations",
            "parameters": {
                "type": "object", 
                "properties": {
                    "data": {"type": "string", "description": "The lookup_sales_data tool's output."},
                    "visualization_goal": {"type": "string", "description": "The goal of the visualization."}
                },
                "required": ["data", "visualization_goal"]
            }
        }
    }
]

# Dictionary mapping function names to their implementations
tool_implementations = {
    "lookup_sales_data": lookup_sales_data,
    "analyze_sales_data": analyze_sales_data, 
    "generate_visualization": generate_visualization
}

In [19]:
def handle_tool_calls(tool_calls, messages):
    for tool in tool_calls:
        function = tool_implementations[tool.function.name]
        function_args = json.loads(tool.function.arguments)
        result = function(**function_args)
        messages.append({"role": "tool", "content": result, "tool_call_id": tool.id})
        
    return messages

In [21]:
SYSTEM_PROMPT = """
You are a helpful assistant that can answer questions about the Store Sales Price Elasticity Promotions dataset.
"""

In [26]:
def run_agent(messages):
    print(f"running agent with messages: {messages}")
    
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
        
    if not any(
        isinstance(messages, dict) and messages.get("role") == "system" for message in messages
    ):
        system_prompt = {"role": "system", "content": SYSTEM_PROMPT}
        messages.append(system_prompt)
        
    while True:
        print("making call to OpenAI")
        response = client.chat.completions.create(
            model = model,
            messages = messages,
            tools = tools,
        )
        messages.append(response.choices[0].message)
        tool_calls = response.choices[0].message.tool_calls
        print("received response with tool calls:", bool(tool_calls))
        
        if tool_calls:
            print("processing tool calls")
            messages = handle_tool_calls(tool_calls, messages)
        else:
            print("no tool calls, returning to final response")
            return response.choices[0].message.content

In [27]:
result = run_agent('Show me the code for graph of sales by store in Nov 2021, and tell me what trends you see.')

running agent with messages: Show me the code for graph of sales by store in Nov 2021, and tell me what trends you see.
making call to OpenAI
received response with tool calls: True
processing tool calls
making call to OpenAI
received response with tool calls: True
processing tool calls
making call to OpenAI
received response with tool calls: True
processing tool calls
making call to OpenAI
received response with tool calls: False
no tool calls, returning to final response


In [28]:
Markdown(result)

### Graph Code for Sales by Store in November 2021

Here is the Python code that generates a visualization of total sales by store for November 2021:

```python
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

data = """Store_Number  Total_Qty_Sold  Total_Sale_Value
0           2970          2312.0      31000.569969
1           2530           890.0      11139.740003
2           3740           949.0      10719.049952
3           4400           225.0       4390.689987
4           1100          1712.0      19298.679917
5           3300          1859.0      23730.719905
6           3190          1003.0      11934.999961
7           2200          1513.0      16854.439921
8           2310          1594.0      16611.539945
9           1870          1648.0      17091.849894
10          3630          1311.0      14202.299958
11          1210          1821.0      21021.669984
12          2750          1462.0      16310.409961
13          1760          1207.0      14598.209948
14          1980           893.0       9084.959929
15           330          1105.0      12569.599911
16          2640          1005.0      10891.639931
17          2090          1080.0      11617.179888
18          4840          1612.0      19127.169971
19          3080          1454.0      18044.450005
20          1320          1701.0      19553.789981
21          1540          1801.0      21207.689942
22          4070           932.0       9161.409921
23          4730           819.0      10571.779962
24           990          1280.0      15101.459945
25          3410          1345.0      15982.159946
26          2420          1370.0      15234.709888
27           660          1208.0      13041.239901
28           770           973.0       9181.769959
29           880          1525.0      17753.769956
30          4180           913.0      10213.819920
31           550           764.0       9554.049975
32          1650          1770.0      23186.909958
"""

df = pd.read_csv(StringIO(data), delim_whitespace=True)

plt.figure(figsize=(10, 6))
plt.bar(df['Store_Number'], df['Total_Sale_Value'], color='skyblue')
plt.title('Total Sales by Store for November 2021')
plt.xlabel('Store Number')
plt.ylabel('Total Sale Value')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
```

### Observed Trends

1. **Sales Performance by Store:**
   - Store 2970 leads in both quantity sold (2312 units) and total sales value ($31,000), indicating strong performance.
   - Store 4400 shows the lowest sales, with 225 units and a total of $4,390, signaling potential issues to address.

2. **Sales Value vs. Quantity Sold:**
   - Certain stores exhibit a high quantity sold with lower sales values, implying a focus on lower-priced products (e.g., Store 2200).
   - Others, like Store 3300, demonstrate a better price point with a significant total sale value.

3. **Diversity in Performance:**
   - Varied sales figures suggest differing operational efficiencies, pricing strategies, or customer bases across stores.
   - Notably high sales values with moderate quantities (e.g., Store 1100) indicate premium product success.

4. **Outliers:**
   - Store 1210 and Store 1540 show exceptional sales performance relative to their quantities sold, which may be worthy of further investigation for practices that could be replicated.

5. **Low-Performing Stores:**
   - Stores like 4730 and 4400 require attention, possibly through enhanced marketing efforts or price adjustments.

6. **Lack of Correlation:**
   - There’s no clear relationship between quantity sold and total sales values, suggesting a mixture of product offerings and sales strategies at play.

### Conclusion
The analysis of the sales data reveals significant variability across stores, presenting opportunities for targeted improvements in marketing, inventory, and pricing in order to enhance overall sales performance. Further qualitative assessments could yield deeper insights into specific drivers behind the numbers.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from io import StringIO

data = """Store_Number  Total_Qty_Sold  Total_Sale_Value
0           2970          2312.0      31000.569969
1           2530           890.0      11139.740003
2           3740           949.0      10719.049952
3           4400           225.0       4390.689987
4           1100          1712.0      19298.679917
5           3300          1859.0      23730.719905
6           3190          1003.0      11934.999961
7           2200          1513.0      16854.439921
8           2310          1594.0      16611.539945
9           1870          1648.0      17091.849894
10          3630          1311.0      14202.299958
11          1210          1821.0      21021.669984
12          2750          1462.0      16310.409961
13          1760          1207.0      14598.209948
14          1980           893.0       9084.959929
15           330          1105.0      12569.599911
16          2640          1005.0      10891.639931
17          2090          1080.0      11617.179888
18          4840          1612.0      19127.169971
19          3080          1454.0      18044.450005
20          1320          1701.0      19553.789981
21          1540          1801.0      21207.689942
22          4070           932.0       9161.409921
23          4730           819.0      10571.779962
24           990          1280.0      15101.459945
25          3410          1345.0      15982.159946
26          2420          1370.0      15234.709888
27           660          1208.0      13041.239901
28           770           973.0       9181.769959
29           880          1525.0      17753.769956
30          4180           913.0      10213.819920
31           550           764.0       9554.049975
32          1650          1770.0      23186.909958
"""

df = pd.read_csv(StringIO(data), delim_whitespace=True)

plt.figure(figsize=(10, 6))
plt.bar(df['Store_Number'], df['Total_Sale_Value'], color='skyblue')
plt.title('Total Sales by Store for November 2021')
plt.xlabel('Store Number')
plt.ylabel('Total Sale Value')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()